In [1]:
from pulp import LpProblem, LpVariable, lpSum, value, LpBinary

class ShiftScheduler:
    def __init__(self, employees, shifts):
        self.employees = employees
        self.shifts = shifts
        self.assignments = None

    def create_lp_problem(self):
        # Create LP problem
        prob = LpProblem("Shift_Scheduling", sense=1)  # sense=1 for minimization

        # Decision variables
        assignments = LpVariable.dicts("Assign", ((employee, shift) for employee in self.employees for shift in self.shifts), 0, 1, LpBinary)

        # Objective function (minimize the total number of shifts assigned)
        prob += lpSum(assignments[employee, shift] for employee in self.employees for shift in self.shifts)

        # Constraints
        # Each shift must be covered
        for shift in self.shifts:
            prob += lpSum(assignments[employee, shift] for employee in self.employees) >= 1

        for employee in self.employees:
            for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]:
                shifts_on_day = [shift for shift in self.shifts if self.shifts[shift].shift_map[day]]
                prob += lpSum(assignments[employee, shift] for shift in shifts_on_day) <= 1
        # Each employee can only work during their available times
        #for employee in self.employees:
        #    for shift in self.shifts:
        #        if not self.is_employee_available(employee, shift):
         #           prob += assignments[employee, shift] == 0

        # Each employee can only be assigned a limited number of shifts (adjust as needed)
        #max_shifts_per_employee = 3
        #for employee in self.employees:
        #    prob += lpSum(assignments[employee, shift] for shift in self.shifts) <= max_shifts_per_employee

        self.assignments = assignments
        return prob

    def is_employee_available(self, employee, shift_name):
        # Check if the employee is available during the shift's time
        shift = self.shifts[shift_name]
        employee_availability = self.employees[employee].date_time.get(shift_name, None)
        if employee_availability:
            shift_start, shift_end = shift.start_time, shift.end_time
            return shift_name in self.employees[employee].date_time and shift_start >= employee_availability[0] and shift_end <= employee_availability[1]
        return False

    def solve_schedule(self):
        prob = self.create_lp_problem()
        prob.solve()

        if prob.status == 1:  # Optimal solution found
            self.print_schedule()

    def print_schedule(self):
        days_of_week = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
        for day in days_of_week:
            print(f"\nAssignments for {day}:")
            for employee in self.employees:
                for shift in self.shifts:
                    if self.shifts[shift].shift_map[day] and value(self.assignments[employee, shift]) == 1:
                        print(f"{employee} is assigned to {shift}")


employee1 = Employee(
    first_name="John",
    last_name="Doe",
    email="john.doe@example.com",
    phone_number="123-456-7890",
    date_time={
        "Monday": ("08:00", "18:00"),
        "Tuesday": ("08:00", "18:00"),
        "Wednesday": ("08:00", "18:00"),
        "Thursday": ("08:00", "18:00"),
        "Friday": ("08:00", "18:00"),
        "Saturday": ("08:00", "18:00"),
        "Sunday": ("08:00", "18:00"),
    }
)

employee2 = Employee(
    first_name="Jane",
    last_name="Smith",
    email="jane.smith@example.com",
    phone_number="987-654-3210",
    date_time={
        "Monday": ("08:00", "18:00"),
        "Tuesday": ("08:00", "18:00"),
        "Wednesday": ("08:00", "18:00"),
        "Thursday": ("08:00", "18:00"),
        "Friday": ("08:00", "18:00"),
        "Saturday": ("08:00", "18:00"),
        "Sunday": ("08:00", "18:00"),
    }
)

shift1 = Shift(
    name="Morning Shift",
    start_time="08:00",
    end_time="12:00",
    shift_map={
        "Monday": True,
        "Tuesday": True,
        "Wednesday": True,
        "Thursday": True,
        "Friday": True,
        "Saturday": False,
        "Sunday": False,
    }
)

shift2 = Shift(
    name="Evening Shift",
    start_time="12:00",
    end_time="14:00",
    shift_map={
        "Monday": True,
        "Tuesday": True,
        "Wednesday": True,
        "Thursday": True,
        "Friday": True,
        "Saturday": True,
        "Sunday": True,
    }
)

employees = {"John": employee1, "Jane": employee2}
shifts = {"Morning Shift": shift1, "Evening Shift": shift2}

scheduler = ShiftScheduler(employees, shifts)
scheduler.solve_schedule()


NameError: name 'Employee' is not defined